### ConversationBufferMemory (Most Basic)

```python
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationChain
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini")

memory = ConversationBufferMemory()

conversation = ConversationChain(
    llm=llm,
    memory=memory,
    verbose=True
)
conversation.predict(input="Hi, my name is Ashik")
conversation.predict(input="What is my name?")

### ConversationBufferWindowMemory

- Stores only last N messages

```python
from langchain.memory import ConversationBufferWindowMemory

memory = ConversationBufferWindowMemory(k=2)

conversation = ConversationChain(
    llm=llm,
    memory=memory,
    verbose=True
)

conversation.predict(input="My name is Ashik")
conversation.predict(input="I live in Kochi")
conversation.predict(input="What is my name?")

### ConversationSummaryMemory

```python
from langchain.memory import ConversationSummaryMemory

memory = ConversationSummaryMemory(llm=llm)

conversation = ConversationChain(
    llm=llm,
    memory=memory,
    verbose=True
)

conversation.predict(input="My name is Ashik")
conversation.predict(input="I am learning LangChain")
conversation.predict(input="What do you know about me?")

### ConversationSummaryBufferMemory
- Hybrid:
   - Recent messages + summary

```python
from langchain.memory import ConversationSummaryBufferMemory

memory = ConversationSummaryBufferMemory(
    llm=llm,
    max_token_limit=100
)

conversation = ConversationChain(
    llm=llm,
    memory=memory,
    verbose=True
)

### Entity Memory
- Tracks specific entities (name, place, etc.)

```python
from langchain.memory import ConversationEntityMemory

memory = ConversationEntityMemory(llm=llm)

conversation = ConversationChain(
    llm=llm,
    memory=memory,
    verbose=True
)

conversation.predict(input="My name is Ashik")
conversation.predict(input="I live in Kochi")
conversation.predict(input="Where do I live?")

___

#### Entity extraction prompt (simplified idea)
- Internally, LangChain uses a prompt similar to:

```raw
You are an AI assistant that extracts entities from conversation.

Conversation:
User: My name is Ashik

What are the entities?
```


- Output:
   - Ashik

- Entity memory structure 
  - {"Ashik": "User's name is Ashik"}

___

### VectorStoreMemory

- Stores memory as embeddings
- Large memory
- Long-term recall

```python
from langchain.memory import VectorStoreRetrieverMemory
from langchain.vectorstores import FAISS
from langchain.embeddings import OpenAIEmbeddings

embedding = OpenAIEmbeddings()

vectorstore = FAISS.from_texts([""], embedding)

retriever = vectorstore.as_retriever()

memory = VectorStoreRetrieverMemory(retriever=retriever)
```

___

```python
from langchain.memory import VectorStoreRetrieverMemory
from langchain.vectorstores import FAISS
from langchain.embeddings import OpenAIEmbeddings
from langchain.chains import ConversationChain
from langchain.llms import OpenAI

# 1. Initialize LLM
llm = OpenAI(temperature=0)

# 2. Initialize embeddings
embedding = OpenAIEmbeddings()

# 3. Create empty vector store
vectorstore = FAISS.from_texts(
    texts=[],  # start empty
    embedding=embedding
)

# 4. Convert to retriever
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 2}  # retrieve top 2 relevant memories
)

# 5. Create memory
memory = VectorStoreRetrieverMemory(
    retriever=retriever
)

# 6. Create conversation chain
conversation = ConversationChain(
    llm=llm,
    memory=memory,
    verbose=True
)

# 7. Interactions
conversation.predict(input="My name is Ashik")
conversation.predict(input="I live in Kochi")
conversation.predict(input="I work as a data engineer")

# 8. Ask something that needs memory
response = conversation.predict(input="Where do I live?")
print(response)

___

### Combined Memory

```python
from langchain.memory import CombinedMemory

memory = CombinedMemory(memories=[memory1, memory2])
```

### Memory with Agents

```python
from langchain.agents import initialize_agent

agent = initialize_agent(
    tools=[],
    llm=llm,
    memory=ConversationBufferMemory(),
    verbose=True
)